In [1]:
import os
import sys
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [2]:
# For Google Colaboratory
import sys, os
if 'google.colab' in sys.modules:
    # mount google drive
    from google.colab import drive
    drive.mount('/content/gdrive')
    path_to_file = '/content/gdrive/My Drive/Guru Innov Challenge'
    print(path_to_file)
    # move to Google Drive directory
    os.chdir(path_to_file)
    !pwd

Mounted at /content/gdrive
/content/gdrive/My Drive/Guru Innov Challenge
/content/gdrive/My Drive/Guru Innov Challenge


In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

Using device: cpu


In [24]:
#pretrained ResNet Model with further data augmentation and fine tuning. up epoch to 7.keep batch size at 6.
import torchvision.models.video as models

DATA_DIR = "/content/gdrive/My Drive/Guru Innov Challenge"
CLIP_LENGTH = 8
BATCH_SIZE = 6
NUM_EPOCHS = 7
LEARNING_RATE = 1e-3

class EmotionsDataset(Dataset):
    def __init__(self, root_dir='data', clip_length=8):
        self.root_dir = root_dir
        self.clip_length = clip_length
        self.samples = []

        self.label_map = {
            'angry': 0,
            'calm': 1,
            'fearful': 2,
            'sad': 3,
            'happy': 4,
            'neutral': 5
        }

        for label_name, label_idx in self.label_map.items():
            subfolder = os.path.join(root_dir, label_name)
            if not os.path.exists(subfolder):
                print(f"Warning: Directory not found: {subfolder}")
                continue
            for fname in os.listdir(subfolder):
                if fname.lower().endswith(('.mp4', '.avi', '.mov')):
                    full_path = os.path.join(subfolder, fname)
                    self.samples.append((full_path, label_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        frames_tensor = self.read_video(video_path)
        return frames_tensor, torch.tensor(label, dtype=torch.long)

    def read_video(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        frames = []
        if total_frames > 0:
          speed_factor = np.random.uniform(0.8, 1.2)  # Mild Time Warping
          new_clip_length = int(self.clip_length * speed_factor)

          # Sample frames randomly
          if total_frames > new_clip_length:
              indices = sorted(np.random.choice(range(total_frames), new_clip_length, replace=False))
          else:
              indices = np.linspace(0, total_frames - 1, new_clip_length).astype(np.int32)

          frame_id = 0
          success = True
          while success:
              success, frame = cap.read()
              if not success:
                  break
              if frame_id in indices:
                  frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

                  # Horizontal Flip (50% chance)
                  if np.random.rand() > 0.5:
                      frame_rgb = cv2.flip(frame_rgb, 1)
                  # Added Random Brightness/Contrast Adjustments
                  if np.random.rand() > 0.5:
                      alpha = 0.8 + np.random.rand() * 0.4
                      beta = np.random.randint(-20, 20)
                      frame_rgb = cv2.convertScaleAbs(frame_rgb, alpha=alpha, beta=beta)

                  frame_rgb = cv2.resize(frame_rgb, (128, 128))
                  frames.append(frame_rgb)
              frame_id += 1

          cap.release()

    # Ensure exactly 8 frames: Trim extra or pad missing frames
        if len(frames) > self.clip_length:
            frames = frames[:self.clip_length]  # Trim
        while len(frames) < self.clip_length:
            frames.append(np.zeros((128, 128, 3), dtype=np.uint8))  # Pad

    # Convert to tensor
        frames_np = np.array(frames, dtype=np.float32) / 255.0
        frames_np = np.transpose(frames_np, (3, 0, 1, 2))  # (channels, clip_len, h, w)

        return torch.from_numpy(frames_np)


In [25]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset

# Load full dataset
full_dataset = EmotionsDataset(root_dir=DATA_DIR, clip_length=CLIP_LENGTH)
# Check dataset size
print(f"Total videos found: {len(full_dataset)}")

# Split into train (80%) and validation (20%)
train_indices, val_indices = train_test_split(range(len(full_dataset)), test_size=0.2, random_state=42)

train_dataset = Subset(full_dataset, train_indices)
val_dataset = Subset(full_dataset, val_indices)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)


Total videos found: 528
Train samples: 422
Validation samples: 106


In [26]:
# Load the pretrained 3D ResNet model
model = models.r3d_18(pretrained=True)

# Modify the final fully connected layer for your number of classes
num_classes = 6  # Change this to the number of classes in your dataset
model.fc = nn.Linear(model.fc.in_features, num_classes)

# Send the model to device (GPU/CPU)
model = model.to(device)

for param in model.parameters():
    param.requires_grad = False  # Freeze all layers

# Only unfreeze the last layer for fine-tuning
for param in model.fc.parameters():
    param.requires_grad = True

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=R3D_18_Weights.KINETICS400_V1`. You can also use `weights=R3D_18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [27]:
# Use Adam optimizer with learning rate and weight decay
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)  # Adjust weight_decay as needed
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5, verbose=True)

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [28]:
# Training loop
for epoch in range(1, NUM_EPOCHS + 1):
    # ----- Training Loop ------
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for videos, labels in train_loader:
        videos = videos.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # Forward pass
        outputs = model(videos)
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * videos.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = 100.0 * correct / total

    # ----- Validation Loop -----
    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for videos, labels in val_loader:
            videos = videos.to(device)
            labels = labels.to(device)

            outputs = model(videos)
            loss = criterion(outputs, labels)
            val_loss_sum += loss.item() * videos.size(0)

            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss_sum / val_total if val_total > 0 else 0.0
    val_acc = 100.0 * val_correct / val_total if val_total > 0 else 0.0

    print(f"Epoch {epoch}: "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%  |  "
          f"Val Loss: {val_loss:.4f},   Val Acc: {val_acc:.2f}%")


Epoch 1: Train Loss: 1.7639, Train Acc: 22.51%  |  Val Loss: 1.6616,   Val Acc: 33.02%
Epoch 2: Train Loss: 1.5874, Train Acc: 36.02%  |  Val Loss: 1.5593,   Val Acc: 33.02%
Epoch 3: Train Loss: 1.4409, Train Acc: 42.18%  |  Val Loss: 1.4157,   Val Acc: 45.28%
Epoch 4: Train Loss: 1.3556, Train Acc: 50.47%  |  Val Loss: 1.2736,   Val Acc: 57.55%
Epoch 5: Train Loss: 1.3310, Train Acc: 47.87%  |  Val Loss: 1.3998,   Val Acc: 42.45%
Epoch 6: Train Loss: 1.2626, Train Acc: 55.69%  |  Val Loss: 1.1423,   Val Acc: 60.38%
Epoch 7: Train Loss: 1.1571, Train Acc: 56.64%  |  Val Loss: 1.1045,   Val Acc: 56.60%


In [29]:
torch.save(model.state_dict(), "6emotions_resnet3dV1.pth")
print("\nModel saved as 6emotions_resnet3dV1.pth")


Model saved as 6emotions_resnet3dV1.pth


In [30]:
import torch
import torch.nn.functional as F

# Emotion label mapping
label_map = {
    0: "angry",
    1: "calm",
    2: "fearful",
    3: "sad",
    4: "happy",
    5: "neutral"
}

def predict_emotion(video_tensor, model, device):
    """
    Given a video tensor, predicts the emotion using the trained model.

    Args:
        video_tensor (torch.Tensor): The input video as a tensor of shape (C, T, H, W).
        model (torch.nn.Module): The trained 3D CNN model.
        device (str): The device ("cpu" or "cuda") to run the inference on.

    Returns:
        str: The detected emotion label.
    """

    model.eval()  # Set model to evaluation mode
    video_tensor = video_tensor.unsqueeze(0).to(device)  # Add batch dimension and move to device

    with torch.no_grad():
        outputs = model(video_tensor)  # Get raw model outputs (logits)
        probabilities = F.softmax(outputs, dim=1)  # Convert logits to probabilities
        predicted_idx = torch.argmax(probabilities, dim=1).item()  # Get highest probability index

        # Check if predicted_idx is within the valid range of label_map
        if 0 <= predicted_idx < len(label_map):
            predicted_emotion = label_map.get(predicted_idx) #use get to safely access, return None if not found
        else:
            predicted_emotion = "Unknown"  # Handle out-of-range predictions
            print(f"Warning: Predicted index {predicted_idx} is outside the valid range.")

    # Convert probabilities to a dictionary with emotion names
    scores = {label_map.get(i, f"class_{i}"): prob.item()
               for i, prob in enumerate(probabilities.squeeze())}


    print(f"Predicted Emotion: {predicted_emotion}")
    print("Scores for each emotion:", scores)

    return predicted_emotion, scores

In [31]:
import cv2
import numpy as np
import torch

def load_video_from_path(video_path, clip_length=8, frame_size=(128, 128)):
    """
    Loads a video file and processes it into a tensor.

    Args:
        video_path (str): Path to the video file.
        clip_length (int): Number of frames to extract.
        frame_size (tuple): Target resize dimensions (height, width).

    Returns:
        torch.Tensor: Video tensor of shape (C, T, H, W).
    """
    cap = cv2.VideoCapture('/content/gdrive/MyDrive/Guru Innov Challenge/test/IMG_0692.MOV')
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    frames = []
    if total_frames > 0:
        indices = np.linspace(0, total_frames - 1, clip_length).astype(np.int32)
        frame_id = 0
        success = True

        while success:
            success, frame = cap.read()
            if not success:
                break
            if frame_id in indices:
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                # Data Augmentation (Optional)
                if np.random.rand() > 0.5:
                    frame_rgb = cv2.flip(frame_rgb, 1)  # Horizontal Flip
                frame_rgb = cv2.resize(frame_rgb, frame_size)  # Resize to match training
                frames.append(frame_rgb)
            frame_id += 1
        cap.release()

    # If the video has fewer frames than clip_length, pad with black frames
    while len(frames) < clip_length:
        frames.append(np.zeros((*frame_size, 3), dtype=np.uint8))

    # Convert to tensor
    frames_np = np.array(frames, dtype=np.float32) / 255.0  # Normalize to [0, 1]
    frames_np = np.transpose(frames_np, (3, 0, 1, 2))  # Convert to (C, T, H, W)

    return torch.from_numpy(frames_np)


In [32]:
# Path to your test video
test_video_path = "/content/gdrive/MyDrive/Guru Innov Challenge/test/IMG_0692.MOV"

# Load video as tensor
video_tensor = load_video_from_path(test_video_path)

# Run prediction
predicted_emotion = predict_emotion(video_tensor, model, device)



Predicted Emotion: sad
Scores for each emotion: {'angry': 0.08205234259366989, 'calm': 0.07648885250091553, 'fearful': 0.36374107003211975, 'sad': 0.3915099501609802, 'happy': 0.07611969858407974, 'neutral': 0.010088068433105946}


In [33]:
NUM_EPOCHS = 10
# Training loop
for epoch in range(1, NUM_EPOCHS + 1):
    # ----- Training Loop ------
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for videos, labels in train_loader:
        videos = videos.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # Forward pass
        outputs = model(videos)
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * videos.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = 100.0 * correct / total

    # ----- Validation Loop -----
    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for videos, labels in val_loader:
            videos = videos.to(device)
            labels = labels.to(device)

            outputs = model(videos)
            loss = criterion(outputs, labels)
            val_loss_sum += loss.item() * videos.size(0)

            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss_sum / val_total if val_total > 0 else 0.0
    val_acc = 100.0 * val_correct / val_total if val_total > 0 else 0.0

    print(f"Epoch {epoch}: "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%  |  "
          f"Val Loss: {val_loss:.4f},   Val Acc: {val_acc:.2f}%")



Epoch 1: Train Loss: 1.1901, Train Acc: 55.45%  |  Val Loss: 1.0781,   Val Acc: 59.43%
Epoch 2: Train Loss: 1.1492, Train Acc: 57.35%  |  Val Loss: 1.0635,   Val Acc: 62.26%
Epoch 3: Train Loss: 1.1078, Train Acc: 60.66%  |  Val Loss: 0.9259,   Val Acc: 66.04%
Epoch 4: Train Loss: 1.0948, Train Acc: 59.48%  |  Val Loss: 1.1155,   Val Acc: 50.00%
Epoch 5: Train Loss: 1.0947, Train Acc: 60.19%  |  Val Loss: 0.9616,   Val Acc: 66.98%
Epoch 6: Train Loss: 1.0294, Train Acc: 61.85%  |  Val Loss: 1.0260,   Val Acc: 59.43%
Epoch 7: Train Loss: 1.0700, Train Acc: 60.90%  |  Val Loss: 0.9522,   Val Acc: 63.21%
Epoch 8: Train Loss: 1.0063, Train Acc: 62.09%  |  Val Loss: 0.9631,   Val Acc: 63.21%
Epoch 9: Train Loss: 1.0296, Train Acc: 60.90%  |  Val Loss: 0.9394,   Val Acc: 64.15%
Epoch 10: Train Loss: 0.9951, Train Acc: 62.80%  |  Val Loss: 0.8964,   Val Acc: 63.21%


In [35]:
torch.save(model.state_dict(), "6emotions_resnet3dV2.pth")
print("\nModel saved as 6emotions_resnet3dV2.pth")


Model saved as 6emotions_resnet3dV2.pth


In [34]:
# Path to your test video
test_video_path = "/content/gdrive/MyDrive/Guru Innov Challenge/test/IMG_0692.MOV"

# Load video as tensor
video_tensor = load_video_from_path(test_video_path)

# Run prediction
predicted_emotion = predict_emotion(video_tensor, model, device)

Predicted Emotion: sad
Scores for each emotion: {'angry': 0.06691256165504456, 'calm': 0.06269873678684235, 'fearful': 0.19146443903446198, 'sad': 0.6355259418487549, 'happy': 0.04105354845523834, 'neutral': 0.0023447570856660604}
